# Extra Model Comparisons


In [ ]:
from pathlib import Path
import json
import warnings
import sys
import pickle

import numpy as np
import pandas as pd
import h5py
from scipy import stats

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

In [ ]:
repo_dir = Path("../../..")

In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))
    
from analysis.curve_fitting.src.fitting_functions import LOSS_FUNCTIONS
from analysis.curve_fitting.src.utils import apply_filters, load_yaml, convert_loss_parameters, convert_loss_parameters_batch

from visualization.src.utils import set_ticks, save_figs, apply_hiearchical_aggregation
from visualization.src.utils import BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, GENERIC_GROUPING_COLUMNS, ARCHITECUTURE_FAMILY_COLORS
from visualization.src.visualize import plot_reg, plot_reg_bivariate, plot_confidence_intervals

In [ ]:
sns.set_theme(style="whitegrid")
sns.set_theme(style="ticks")

artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir = repo_dir / "visualization" / "paper" / "supp" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"


In [ ]:
results_file = artifacts_dir / "pretraining_results_with_metadata.csv"
df_all = pd.read_csv(results_file, low_memory=False)

df_all.shape


In [ ]:
METRIC = "pearsonr_nc"

BENCHMARKS = [
    "bs_fz",
    "bs_mh",
    "tvsd",
    "things_fmri",
    "nsd_func1pt8mm_individualROIs",
    "things_eeg1",
    "things_eeg2",
    "things_meg",
]
BENCHMARKS_WITH_AVERAGE = BENCHMARKS + ["benchmark_average"]
BENCHMARK_ORDER = {benchmark: idx for idx, benchmark in enumerate(BENCHMARKS_WITH_AVERAGE)}

SELECTED_MODELS = [
    "adv_resnet152_imagenet_full_ffgsm_eps-1_alpha-125-ep10_seed-0",
    "resnet152_imagenet_full",
    "alexnet_imagenet_full_seed-0",
    "cornet_s_imagenet_full_seed-0",
    "deit_small_imagenet_full_seed-0",
    "convnext_xxlarge.clip_laion2b_soup_ft_in1k",
    "vit_base_patch16_dinov3.lvd1689m",
    "vjepa2-vitl-fpc64-256",
    "Qwen3-VL-2B-Instruct",
]

MODEL_NAME_MAP = {
    "adv_resnet152_imagenet_full_ffgsm_eps-1_alpha-125-ep10_seed-0": "Adv. ResNet-152",
    "resnet152_imagenet_full": "ResNet-152",
    "alexnet_imagenet_full_seed-0": "AlexNet",
    "cornet_s_imagenet_full_seed-0": "CORnet-S",
    "deit_small_imagenet_full_seed-0": "ViT-S",
    "convnext_xxlarge.clip_laion2b_soup_ft_in1k": "ConvNeXt-XXL CLIP",
    "vit_base_patch16_dinov3.lvd1689m": "DINOv3 ViT-B",
    "vjepa2-vitl-fpc64-256": "V-JEPA2 ViT-L",
    "Qwen3-VL-2B-Instruct": "Qwen3-VL 2B",
}

FAMILY_COLOR_OVERRIDES = {
    "Qwen3-VL": "#457B9D",
    "VJEPA2": "#2A9D8F",
}


In [ ]:
def lighten_color(color: str, amount: float = 0.35) -> str:
    rgb = np.array(mcolors.to_rgb(color))
    return mcolors.to_hex(rgb + (1 - rgb) * amount)


def get_family_color(family: str) -> str:
    if family in ARCHITECUTURE_FAMILY_COLORS:
        return ARCHITECUTURE_FAMILY_COLORS[family]
    return FAMILY_COLOR_OVERRIDES.get(family, "#6C757D")


def style_axis(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    return ax


In [ ]:
grouping_cols = [
    "model_id",
    "backbone_source",
    "backbone_arch",
    "backbone_n_params",
    "backbone_macs",
    "backbone_flops",
    "backbone_arch_family",
]

df_agg = apply_hiearchical_aggregation(
    df=df_all,
    groupby_cols=grouping_cols + ["benchmark_name"],
    agg_cols=METRICS_COMPACT,
    agg_func="mean",
)

df_agg_avg = df_agg.groupby(grouping_cols).agg(
    {metric: "mean" for metric in METRICS_COMPACT}
).reset_index()
df_agg_avg["benchmark_name"] = "benchmark_average"

In [ ]:
df_agg = pd.concat([df_agg, df_agg_avg], ignore_index=True)
df_agg = df_agg[df_agg["model_id"].isin(SELECTED_MODELS)].copy()
df_agg["model_name"] = df_agg["model_id"].map(MODEL_NAME_MAP)
df_agg["benchmark_label"] = df_agg["benchmark_name"].map(BENCHMARK_NAME_MAPPING)
df_agg["benchmark_order"] = df_agg["benchmark_name"].map(BENCHMARK_ORDER)
df_agg["family_color"] = df_agg["backbone_arch_family"].map(get_family_color)

missing_models = sorted(set(SELECTED_MODELS) - set(df_agg["model_id"].unique()))
assert not missing_models, f"Selected models missing after aggregation: {missing_models}"

In [ ]:
df_average = df_agg[df_agg["benchmark_name"] == "benchmark_average"].copy()
model_order = [MODEL_NAME_MAP[model_id] for model_id in SELECTED_MODELS]
model_color_map = {
    name: df_average.set_index("model_name").loc[name, "family_color"]
    for name in model_order
}


df_agg["model_name"] = pd.Categorical(df_agg["model_name"], model_order, ordered=True)
df_agg = df_agg.sort_values(["benchmark_order", "model_name"])

df_average[["model_name", "backbone_arch_family", "backbone_n_params", METRIC]].sort_values(METRIC, ascending=False)


## Grouped Bar Plot


In [ ]:
figsize = (18, 6)
zoom = 0.6
zoom = 0.75
figsize_zoomed = (figsize[0] * zoom, figsize[1] * zoom)

fig, ax = plt.subplots(figsize=figsize_zoomed, dpi=300)


df = df_agg.copy()
df["model_name"] = pd.Categorical(
    df["model_name"].astype(str),
    model_order,
    ordered=True,
)
df["benchmark_name"] = pd.Categorical(
    df["benchmark_name"],
    BENCHMARKS_WITH_AVERAGE,
    ordered=True,
)
df = df.sort_values(["benchmark_name", "model_name"])



sns.barplot(
    data=df,
    x="benchmark_label",
    y=METRIC,
    hue="model_name",
    order=[BENCHMARK_NAME_MAPPING[benchmark] for benchmark in BENCHMARKS_WITH_AVERAGE],
    hue_order=model_order,
    palette=[model_color_map[model] for model in model_order],
    edgecolor="0.2",
    linewidth=0.35,
    ax=ax,
)
ax.set_title("Selected Extra Models Across Benchmarks", fontsize=15, fontweight="bold")
ax.set_xlabel("Benchmark", fontsize=12, fontweight="bold")
ax.set_ylabel("Pearson r (noise-corrected)", fontsize=12, fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_ylim(0, 1)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, axis='y', linestyle='--', alpha=0.35)
ax.legend(title="Model", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
fig.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir,
    base_filename="model_extra-grouped_bar",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: Grouped Bar Plot**

Supplementary Fig. S\*: Bars show alignment for each selected model across benchmarks, including the benchmark-average column.


## Per-Benchmark Bar Grid

### Vertical

In [ ]:
zoom = 0.75
figsize = (30, 24)
figsize_zoomed = (figsize[0] * zoom, figsize[1] * zoom)
fig, axes = plt.subplots(3, 3, figsize=figsize_zoomed, dpi=500)
axes = axes.flatten()

df = df_agg.copy()
df["model_name"] = pd.Categorical(
    df["model_name"].astype(str),
    model_order,
    ordered=True,
)



for idx, (ax, benchmark) in enumerate(zip(axes, BENCHMARKS_WITH_AVERAGE)):
    data = df[df["benchmark_name"] == benchmark].copy()
    data = data.sort_values("model_name")
    sns.barplot(
        data=data,
        x="model_name",
        y=METRIC,
        hue="model_name",
        order=model_order,
        hue_order=model_order,
        palette=[model_color_map[model] for model in model_order],
        edgecolor="0.2",
        linewidth=0.35,
        legend=False,
        ax=ax,
    )
    ax.set_title(BENCHMARK_NAME_MAPPING[benchmark], fontsize=16, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Alignment" if idx % 3 == 0 else "", fontsize=12, fontweight="bold")
    ax.set_ylim(data[METRIC].min() *0.99, data[METRIC].max() *1.01)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=12)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linestyle="--", alpha=0.35)

# fig.suptitle("Selected Extra Models by Benchmark", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir,
    base_filename="model_extra-per_benchmark-vertical",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


### Horizontal

In [ ]:
zoom = 0.75
figsize = (30, 24)

zoom = 0.5
figsize = (24, 21)


figsize_zoomed = (figsize[0] * zoom, figsize[1] * zoom)
fig, axes = plt.subplots(3, 3, figsize=figsize_zoomed, dpi=500)
axes = axes.flatten()

for idx, (ax, benchmark) in enumerate(zip(axes, BENCHMARKS_WITH_AVERAGE)):
    data = df_agg[df_agg["benchmark_name"] == benchmark].copy()
    data = data.set_index("model_name").loc[model_order].reset_index()
    y = np.arange(len(model_order))
    bar_colors = [model_color_map[str(model)] for model in data["model_name"]]
    ax.barh(y, data[METRIC], color=bar_colors, edgecolor="0.2", linewidth=0.5)
    ax.set_title(BENCHMARK_NAME_MAPPING[benchmark], fontsize=12.5, fontweight="bold")
    ax.set_yticks(y)
    if idx % 3 == 0:
        ax.set_yticklabels(data["model_name"], fontsize=9)
    else:
        ax.set_yticklabels([])
    ax.invert_yaxis()
    ax.set_xlabel("Alignment" if idx >= 6 else "", fontsize=12, fontweight="bold"   )
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", linestyle="--", alpha=0.35)
    ax.set_xlim(data[METRIC].min() *0.99, data[METRIC].max() *1.01)
    


# fig.suptitle("Per-Benchmark Extra Model Rankings", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir,
    base_filename="model_extra-per_benchmark-horizontal",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: Per-Benchmark Bar Grid**

Supplementary Fig. S\*: Each panel shows alignment for all selected extra models on one benchmark.


## Model-by-Benchmark Heatmap

The heatmap makes the per-benchmark structure easier to scan than grouped bars.


In [ ]:
heatmap_data = df_agg.pivot_table(
    index="benchmark_label",
    columns="model_name",
    values=METRIC,
    aggfunc="mean",
).loc[
    [BENCHMARK_NAME_MAPPING[benchmark] for benchmark in BENCHMARKS_WITH_AVERAGE],
    model_order,
]

fig, ax = plt.subplots(figsize=(10.5, 5.4), dpi=300)
hm = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".3f",
    cmap="YlGnBu",
    linewidths=0.5,
    linecolor="white",
    ax=ax,
)
cbar = hm.collections[0].colorbar
cbar.set_label("Pearson r (noise-corrected)", fontweight="bold", fontsize=12)

ax.set_title("Alignment by Benchmark and Model", fontsize=15, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
fig.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir,
    base_filename="model_extra-heatmap",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: Extra Model Benchmark Heatmap**

Supplementary Fig. S\*: Alignment scores for each selected model and benchmark. Rows are evaluation benchmarks and columns are models ordered by benchmark-average alignment.


## Model Scale vs Alignment

This additional plot asks whether the strongest extra models are simply the largest models.


In [ ]:
scale_df = df_average.copy()
scale_df["model_name"] = pd.Categorical(scale_df["model_name"], model_order, ordered=True)
scale_df = scale_df.sort_values("model_name")

fig, ax = plt.subplots(figsize=(7.2, 5.0), dpi=300)
for _, row in scale_df.iterrows():
    color = model_color_map[row["model_name"]]
    ax.scatter(
        row["backbone_n_params"],
        row[METRIC],
        s=95,
        color=color,
        edgecolor="0.2",
        linewidth=0.6,
        zorder=3,
    )
    ax.annotate(
        row["model_name"],
        (row["backbone_n_params"], row[METRIC]),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=8.5,
    )

ax.set_xscale("log")
ax.set_title("Model Size vs Alignment", fontsize=15, fontweight="bold")
ax.set_xlabel("Backbone Parameters (log scale)", fontsize=12, fontweight="bold")
ax.set_ylabel("Alignment (NC Pearson r)", fontsize=12, fontweight="bold")
ax.grid(True, which="both", axis="both", linestyle="--", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir,
    base_filename="model_extra-params_vs_alignment",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: Model Scale vs Alignment**

Supplementary Fig. S\*: Relationship between model parameter count and benchmark-average alignment for the selected extra models. The plot separates model scale from empirical alignment and highlights cases where larger parameter count does not directly imply higher neural alignment.
